In [ ]:
# If you wish to environment variables contained in a `.env` file.
# Comment these lines if you do not need so.
%pip install python-dotenv
%load_ext dotenv
%dotenv


In [ ]:
import os

# Either add the path here replacing `None` or through the environment variable ICEDYN_DATA
ICEDYN_DATA = None
ICEDYN_DATA = ICEDYN_DATA or os.environ.get('ICEDYN_DATA')

# Since we downloaded all files in one folder, let's add some metadata
# indicating what these files are and what versions of the dataset used.
metadata_text = """\
Files downloaded from MIMIC-III v1.4:
    PATIENTS.csv.gz ADMISSIONS.csv.gz DIAGNOSES_ICD.csv.gz D_ICD_DIAGNOSES.csv.gz
Files downloaded from MIMIC-IV v3.1:
    patients.csv.gz admissions.csv.gz diagnoses_icd.csv.gz d_icd_diagnoses.csv.gz
"""
%alias echo echo
%echo f"{metadata_text}" > {ICEDYN_DATA}/README.txt

%ls -hs {ICEDYN_DATA} | grep -E "(csv|txt)"
# If you execute this cell, should see the (4 + 4 + 1 = 9) files listed below.


In [ ]:
import logging
import os
import shutil

# We write the low-level log into the file data_proc.log in the current directory.
if os.path.isdir('logs'):
    shutil.rmtree('logs')
os.makedirs('logs')
logging.basicConfig(format='%(asctime)s,%(msecs)03d %(levelname)-8s [%(filename)s:%(lineno)d] %(message)s',
                    datefmt='%Y-%m-%dT%H:%M:%S',
                    encoding='utf-8', level=logging.DEBUG,
                    filename='logs/data_proc.log', filemode='w')

import ehrax as rx
from ehrax.example_datasets import study_mimic_dx_summary as rxd


In [ ]:
%%time
m3_dataset_init, m3_schemes = rxd.mimiciii_from_paths(patients=f"{ICEDYN_DATA}/PATIENTS.csv.gz",
                                                      admissions=f"{ICEDYN_DATA}/ADMISSIONS.csv.gz",
                                                      diagnoses_icd=f"{ICEDYN_DATA}/DIAGNOSES_ICD.csv.gz",
                                                      d_icd_diagnoses=f"{ICEDYN_DATA}/D_ICD_DIAGNOSES.csv.gz", )

m4_dataset_init, m4_schemes = rxd.mimiciv_from_paths(patients=f"{ICEDYN_DATA}/patients.csv.gz",
                                                     admissions=f"{ICEDYN_DATA}/admissions.csv.gz",
                                                     diagnoses_icd=f"{ICEDYN_DATA}/diagnoses_icd.csv.gz",
                                                     d_icd_diagnoses=f"{ICEDYN_DATA}/d_icd_diagnoses.csv.gz", )



In [ ]:
%%time

# Combine both coding schemes managers into one manager
mimic_schemes = m3_schemes + m4_schemes
dataset_pipeline = rxd.default_dataset_pipeline()
m3_dataset = m3_dataset_init.execute_pipeline(dataset_pipeline, mimic_schemes)
m4_dataset = m4_dataset_init.execute_pipeline(dataset_pipeline, mimic_schemes)


In [ ]:
%%time
m3_dataset = m3_dataset_init.execute_pipeline(dataset_pipeline, mimic_schemes)
m4_dataset = m4_dataset_init.execute_pipeline(dataset_pipeline, mimic_schemes)


In [ ]:
# schemes_0 (contains a mapping between mixed_scheme(dx_icd9, dx_icd10) to dx_flat_ccs directly through the tables from AHRQ
schemes_0 = mimic_schemes
m_stats0 = rx.Dataset.two_stats(m3_dataset, m4_dataset, coding_schemes_manager=schemes_0)
p_tests0 = m_stats0.target_p_tests.outcome('dx_flat_ccs_v1', 'admission')
p_summary0 = m_stats0.summerise_p_tests(p_tests0)


In [ ]:
# schemes_1 (contains a mapping between the mixed_scheme(dx_icd9, dx_icd10) to dx_flat_ccs indirectly through dx_icd9.
schemes_1 = mimic_schemes
source = m4_dataset.config.scheme.dx_discharge
target = mimic_schemes.outcome_data['dx_flat_ccs_v1'].base_name
schemes_1 = schemes_1.add_chained_map(source, 'dx_icd9', target, overwrite=True)
m_stats1 = rx.Dataset.two_stats(m3_dataset, m4_dataset, coding_schemes_manager=schemes_1)
p_tests1 = m_stats1.target_p_tests.outcome('dx_flat_ccs_v1', 'admission')
p_summary1 = m_stats1.summerise_p_tests(p_tests1)


In [ ]:
import pandas as pd
pd.concat([p_summary0, p_summary1], axis=1)
